In [1]:
import sqlite3
import torch
import numpy as np

# THIS IS TO INITIALIZE THE DATABASE FOR THE FIRST TIME ONLY


def init_memory_block(db_path='brain.db'):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    
    # 1. Weights Table: layer_name | binary_blob | shape
    c.execute("CREATE TABLE IF NOT EXISTS model_store (layer TEXT, blob BLOB, shape TEXT)")
    
    # 2. Logging Table: binary_drawing | predicted_label
    c.execute("CREATE TABLE IF NOT EXISTS drawing_logs (image_blob BLOB, guess TEXT)")

    # 3. Seed initial random weights (784 pixels -> 26 letters)
    c.execute("SELECT COUNT(*) FROM model_store")
    if c.fetchone()[0] == 0:
        weights = torch.randn(26, 784).numpy().astype(np.float32)
        c.execute("INSERT INTO model_store VALUES (?, ?, ?)", 
                  ('fc_weight', weights.tobytes(), str(weights.shape)))
        print("✅ Weights seeded into Memory Block.")
    
    conn.commit()
    conn.close()

if __name__ == "__main__":
    init_memory_block()


✅ Weights seeded into Memory Block.


In [3]:
import sqlite3
import numpy as np

def view_map(db_path='brain.db'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    for table in ['model_store', 'drawing_logs']:
        print(f"\n--- Table: {table} ---")
        cursor.execute(f"SELECT * FROM {table}")
        rows = cursor.fetchall()
        
        if not rows:
            print("Empty.")
            continue

        for i, row in enumerate(rows[:5]): # Show first 5 entries
            print(f"Entry {i}:")
            for val in row:
                if isinstance(val, bytes):
                    # Fixed calculation: 4 bytes per float32
                    print(f"  -> BLOB: {len(val)} bytes ({len(val)//4} floats)")
                else:
                    print(f"  -> Data: {val}")
    conn.close()

if __name__ == "__main__":
    view_map()



--- Table: model_store ---
Entry 0:
  -> Data: fc_weight
  -> BLOB: 81536 bytes (20384 floats)
  -> Data: (26, 784)

--- Table: drawing_logs ---
Entry 0:
  -> BLOB: 3136 bytes (784 floats)
  -> Data: A
Entry 1:
  -> BLOB: 3136 bytes (784 floats)
  -> Data: C
Entry 2:
  -> BLOB: 3136 bytes (784 floats)
  -> Data: K
Entry 3:
  -> BLOB: 3136 bytes (784 floats)
  -> Data: W
Entry 4:
  -> BLOB: 3136 bytes (784 floats)
  -> Data: W


In [2]:
from flask import Flask, request, jsonify, render_template_string
import sqlite3
import numpy as np
import base64
from io import BytesIO
from PIL import Image

app = Flask(__name__)

# Minimal HTML inside the script for "Final Draft" portability
HTML = """
<!DOCTYPE html>
<html>
<body style="background: #222; color: white; text-align: center;">
    <canvas id="c" width="280" height="280" style="border: 2px solid white; background: black; cursor: crosshair;"></canvas>
    <br><button onclick="send()" style="padding: 10px 20px; margin: 10px;">Predict</button>
    <button onclick="ctx.clearRect(0,0,280,280); ctx.beginPath();" style="padding: 10px 20px;">Clear</button>
    <h1 id="r">Draw a Letter</h1>
    <script>
        const c = document.getElementById('c'); const ctx = c.getContext('2d');
        ctx.strokeStyle = "white"; ctx.lineWidth = 15; let d = false;
        c.onmousedown = () => { d = true; ctx.beginPath(); };
        c.onmouseup = () => d = false;
        c.onmousemove = (e) => { if(d) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); } };
        async function send() {
            const img = c.toDataURL('image/png');
            const res = await fetch('/predict', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({img}) });
            const data = await res.json(); document.getElementById('r').innerText = "Result: " + data.pred;
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index(): return render_template_string(HTML)

@app.route('/predict', methods=['POST'])
def predict():
    # 1. Process Drawing
    data = request.json['img'].split(',')[1]
    img = Image.open(BytesIO(base64.b64decode(data))).convert('L').resize((28, 28))
    img_flat = np.array(img).flatten().astype(np.float32) / 255.0

    # 2. Access Memory Block
    conn = sqlite3.connect('brain.db')
    row = conn.execute("SELECT blob, shape FROM model_store WHERE layer='fc_weight'").fetchone()
    weights = np.frombuffer(row[0], dtype=np.float32).reshape(eval(row[1]))

    # 3. Compute Result
    pred_idx = np.argmax(np.dot(weights, img_flat))
    letter = chr(65 + pred_idx)

    # 4. Correct Log: Store the 28x28 drawing (~3136 bytes), NOT the weights!
    conn.execute("INSERT INTO drawing_logs VALUES (?, ?)", (img_flat.tobytes(), letter))
    conn.commit()
    conn.close()

    return jsonify({'pred': letter})

if __name__ == '__main__':
    app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Apr/2026 10:49:10] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:49:10] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [07/Apr/2026 10:49:15] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:49:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:49:52] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:50:04] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:50:10] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:50:12] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:50:13] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:50:20] "POST /predict HTTP/1.1" 200 -
